# Open RAG From Scratch — Parts 1–4

This notebook builds a retrieval-augmented generation (RAG) chain **one cell at a time**.

## Setup

```
.venv\Scripts\python.exe -m pip install -r requirements.txt
```

The one piece installed separately is **Ollama**, the local model runtime (install once, then `ollama pull gpt-oss:20b`).

### Imports and configuration

All tunable parameters live in one `CONFIG` dict, so experiments never require editing pipeline code. The comments mark which knobs are actually used in this notebook.


In [1]:
import requests
import trafilatura
import chromadb
import numpy as np
from collections import Counter

from tokenizers import Tokenizer                       # raw HF tokenizer (for counting)
from transformers import AutoTokenizer                  # HF fast tokenizer (for chonkie)
from chonkie import RecursiveChunker, TokenChunker
from chonkie.tokenizer import TransformersAutoTokenizer
from sentence_transformers import SentenceTransformer
import ollama

print('imports ok')

c:\dev\open-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports ok


In [2]:
CONFIG = {
    # --- Indexing ---
    'chunk_size_tokens': 300,        # used: max tokens per chunk (token-based chunker)
    'chunk_overlap_tokens': 50,      # used: overlap for the overlap demo
    'embed_model': 'Qwen/Qwen3-Embedding-0.6B',
    'store_path': './index/chroma',
    'collection': 'docs',
    # --- Retrieval ---
    'k': 5,                          # used: top-k retrieved per query
    # --- Generation ---
    'llm_model': 'gpt-oss:20b',
    'temperature': 0.0,
    'context_window': 32768,         # used: context-budget check
    'max_answer_tokens': 1024,       # reserved space for the answer
}
CONFIG

{'chunk_size_tokens': 300,
 'chunk_overlap_tokens': 50,
 'embed_model': 'Qwen/Qwen3-Embedding-0.6B',
 'store_path': './index/chroma',
 'collection': 'docs',
 'k': 5,
 'llm_model': 'gpt-oss:20b',
 'temperature': 0.0,
 'context_window': 32768,
 'max_answer_tokens': 1024}

### Check the local model runtime

Generation needs a model server running on your machine. This confirms `ollama` is up and the configured model has been pulled.


In [3]:
try:
    listing = ollama.list()
    names = [m.get('name', m.get('model', '')) for m in listing.get('models', [])]
    print('Ollama is running. Available models:')
    for n in names:
        print(' -', n)
    present = any(CONFIG['llm_model'].split(':')[0] in n for n in names)
    print('Configured model present:', present)
    if not present:
        print('Pull it with:  ollama pull', CONFIG['llm_model'])
except Exception as e:
    print('Could not reach Ollama:', e)
    print('Start it with:  ollama serve   then   ollama pull', CONFIG['llm_model'])

Ollama is running. Available models:
 - gpt-oss:20b
Configured model present: True


# Part 1 — Overview

Before taking the chain apart, run the whole thing in a single cell so the shape of the system is clear. Read left to right: **load** a page, **split** it into chunks, **embed + store** them, **retrieve** the chunks relevant to a question, and **generate** an answer from them.

This is deliberately compact — the later parts rebuild each step with explanation and observation.


In [4]:
# Load the embedding model once; it is reused for the rest of the notebook.
model = SentenceTransformer(CONFIG['embed_model'])

URL = 'https://lilianweng.github.io/posts/2023-06-23-agent/'
source_id = 'lilianweng-agent'
question = 'What is Task Decomposition?'

# ---- INDEX (offline) ----
text = trafilatura.extract(requests.get(URL).text)

auto_tok = AutoTokenizer.from_pretrained(CONFIG['embed_model'])
chonkie_tok = TransformersAutoTokenizer(auto_tok)
chunker = RecursiveChunker(tokenizer=chonkie_tok, chunk_size=CONFIG['chunk_size_tokens'])
passages = [chunk.text for chunk in chunker.chunk(text)]

vectors = model.encode(passages, normalize_embeddings=True)

client = chromadb.PersistentClient(path=CONFIG['store_path'])
collection = client.get_or_create_collection(
    name=CONFIG['collection'], metadata={'hnsw:space': 'cosine'}
)
collection.upsert(
    ids=[f'{source_id}:{i}' for i in range(len(passages))],
    documents=passages,
    embeddings=vectors.tolist(),
    metadatas=[{'source_id': source_id, 'chunk_index': i} for i in range(len(passages))],
)

# ---- RETRIEVE (online) ----
query_vector = model.encode([question], normalize_embeddings=True).tolist()
hits = collection.query(query_embeddings=query_vector, n_results=CONFIG['k'])
context = '\n\n'.join(hits['documents'][0])

# ---- GENERATE ----
prompt = f'''Answer the question based only on the following context:
{context}

Question: {question}
'''
reply = ollama.chat(
    model=CONFIG['llm_model'],
    options={'temperature': CONFIG['temperature']},
    messages=[{'role': 'user', 'content': prompt}],
)
print(reply['message']['content'])

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 940.51it/s]


{
    "thoughts": {
        "text": "Task Decomposition is the process of breaking a complex task into smaller, manageable sub‑tasks or steps, enabling clearer planning, execution, and evaluation.",
        "plan": "- Identify the overall goal\n- Enumerate sub‑goals or intermediate steps\n- Determine the order or parallelism of steps\n- Assign responsibilities or tools to each step\n- Review and refine the decomposition as needed",
        "criticism": "Make sure the decomposition captures all essential aspects of the task without oversimplifying or omitting critical dependencies.",
        "speak": "Task Decomposition involves breaking a complex task into smaller, manageable steps."
    },
    "command": {
        "name": "do_nothing",
        "args": {}
    }
}


That single cell is the whole of RAG. Notice the two halves: **indexing** happened once (load, split, embed, store), and only the last three steps — embed the question, retrieve, generate — run per question.

Now we take it apart, one idea at a time.

# Part 2 — Indexing

Indexing prepares your documents so they can be searched by meaning. It runs once, ahead of time, and has four steps: **load → split → embed → store**.

We start with the smallest possible example — two toy sentences — so the mechanics of embedding and similarity are easy to see, and then scale up to a real article.


## 2.1 Documents

A **document** is just the raw text you want the system to know about — a web page, a note, a file. At this stage it is text plus a little metadata (where it came from). We begin with the smallest possible example.


In [5]:
question = 'What kinds of pets do I like?'
document = 'My favorite pet is a cat.'

print('question:', question)
print('document:', document)

question: What kinds of pets do I like?
document: My favorite pet is a cat.


## 2.2 Token counting

Models don't read characters or whole words — they read **tokens** (word pieces). Counting tokens tells us how much text fits in the model's **context window**, and how big our chunks should be.

One rule matters: use the **model's own tokenizer**. A count from a different tokenizer will mis-size chunks. We load the tokenizer that belongs to our embedding model and count with it.


In [6]:
# Two tokenizers, two jobs (both wrap the *same* model vocabulary):
#   hf_tok     -> the raw fast tokenizer, used by our count_tokens() helper below
#   auto_tok   -> the transformers wrapper, which is what chonkie expects
#                 (a raw tokenizers.Tokenizer is not accepted by chonkie 1.7.0)
hf_tok = Tokenizer.from_pretrained(CONFIG['embed_model'])
auto_tok = AutoTokenizer.from_pretrained(CONFIG['embed_model'])

def count_tokens(text):
    return len(hf_tok.encode(text).ids)

print('tokens in document:', count_tokens(document))
print('tokens in question:', count_tokens(question))

tokens in document: 8
tokens in question: 9


**Observation.** The same sentence in two tokenizers would give two different counts — that mismatch is exactly why we always use the target model's tokenizer. A useful rule of thumb for English is ~4 characters per token.


## 2.3 Embeddings

An **embedding** turns text into a list of numbers — a **vector** — that captures its meaning. Texts with similar meaning end up as nearby vectors. This is what makes search **semantic** rather than a literal word match.

Every model produces a vector of a fixed length; ours gives **1024 numbers**.


In [7]:
# Embed both toy texts; each becomes a fixed-length vector.
document_vector = model.encode([document], normalize_embeddings=True)[0]
question_vector = model.encode([question], normalize_embeddings=True)[0]

print('vector length:', len(document_vector))
print('first 10 numbers:', np.round(document_vector[:10], 3))

vector length: 1024
first 10 numbers: [ 0.024  0.039  0.002 -0.079 -0.054  0.003  0.021  0.021  0.029  0.098]


In [8]:
# Contrast: a relevant, a tangential, and an unrelated sentence.
samples = {
    'relevant   ': 'My favorite pet is a cat.',
    'tangential ': 'Pets need food and daily care.',
    'unrelated  ': 'The capital of France is Paris.',
}
sample_vectors = {k: model.encode([v], normalize_embeddings=True)[0] for k, v in samples.items()}

print('cosine similarity of each with the question:')
for k, v in samples.items():
    s = float(np.dot(question_vector, sample_vectors[k]))
    print(f'  {k} {v!r:<40} -> {s:.3f}')

cosine similarity of each with the question:
  relevant    'My favorite pet is a cat.'              -> 0.560
  tangential  'Pets need food and daily care.'         -> 0.476
  unrelated   'The capital of France is Paris.'        -> 0.112


**Observation.** The relevant sentence scores highest, the tangential one sits in the middle, and the unrelated one is near the bottom. Notice *why* the middle one loses: it repeats the question's keyword "pet" but does not answer the question. The ranking is decided by meaning, not by word overlap — that gap is the whole idea behind semantic search.


## 2.4 Cosine similarity

To compare two embeddings we measure the **angle** between them — cosine similarity. A score near `1` means very similar meaning; near `0` means unrelated. Since the vectors are normalised, the dot product *is* the cosine similarity.

### The whole idea in miniature

Here is the entire RAG idea in miniature: put two documents in a tiny store, embed the question, and find the nearest. This is exactly what Part 3 does — just at toy scale.


In [9]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

score = cosine_similarity(question_vector, document_vector)
print('cosine similarity (question vs document):', round(score, 3))

# --- miniature store: two documents, embedded and searchable ---
mini_texts = [document, 'The capital of France is Paris.']
mini_vectors = model.encode(mini_texts, normalize_embeddings=True)
mini = chromadb.EphemeralClient()
mini_col = mini.get_or_create_collection('mini', metadata={'hnsw:space': 'cosine'})
mini_col.upsert(
    ids=['d0', 'd1'],
    documents=mini_texts,
    embeddings=mini_vectors.tolist(),
    metadatas=[{'source_id': 'doc-a'}, {'source_id': 'doc-b'}],
)

mini_q = model.encode([question], normalize_embeddings=True).tolist()
mini_hits = mini_col.query(query_embeddings=mini_q, n_results=2)
print('\nmini retrieval, k=2:')
for i, (cid, doc, dist) in enumerate(zip(mini_hits['ids'][0], mini_hits['documents'][0], mini_hits['distances'][0]), 1):
    print(f'  [{i}] id={cid}  score={1 - dist:.3f}  {doc!r}')

cosine similarity (question vs document): 0.56

mini retrieval, k=2:
  [1] id=d0  score=0.560  'My favorite pet is a cat.'
  [2] id=d1  score=0.112  'The capital of France is Paris.'


**Observation.** The pet document ranks first even though the other document shares no words with the question. Tiny store, same mechanism, real result.


## 2.5 Document loaders

Real sources are messy — a web page is mostly menus, ads and footers wrapped around a small article. A **loader** fetches the page and keeps only the meaningful text, so noise never reaches our embeddings.

`trafilatura` extracts the main article automatically. We record a `source_id` so every chunk can be traced back to its source, and answers can cite it.


In [10]:
source_id = 'lilianweng-agent'

raw_html = requests.get(URL).text
text = trafilatura.extract(raw_html)

print('characters extracted:', len(text))
print('tokens in article   :', count_tokens(text))
print('preview:', text[:300].replace(chr(10), ' '), '...')

characters extracted: 39316
tokens in article   : 8793
preview: Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it c ...


## 2.6 Splitting (chunking)

Long documents are cut into smaller **chunks**. A chunk is the unit that gets embedded, retrieved and shown to the model, so its size matters. Small chunks give precise matches; large chunks carry more context but dilute relevance.

Here we split by **tokens** (using the model's tokenizer), which is the honest unit — recall the theory's rule: *chunk sizing is token-based, not character-based*.


In [11]:
# Token-based recursive chunking with the model's own tokenizer.
chonkie_tok = TransformersAutoTokenizer(auto_tok)
chunker = RecursiveChunker(tokenizer=chonkie_tok, chunk_size=CONFIG['chunk_size_tokens'])
chunks = chunker.chunk(text)
passages = [chunk.text for chunk in chunks]

print('number of chunks:', len(passages))

number of chunks: 34


In [12]:
# Chunk census: are the chunks reasonably sized?
sizes = [count_tokens(p) for p in passages]
print(f'chunks          : {len(sizes)}')
print(f'tokens min/mean/max: {min(sizes)} / {round(np.mean(sizes),1)} / {max(sizes)}')
print(f'total tokens    : {sum(sizes)}  (~{sum(sizes)/CONFIG["context_window"]*100:.1f}% of the context window for the whole article)')

chunks          : 34
tokens min/mean/max: 23 / 259.6 / 301
total tokens    : 8827  (~26.9% of the context window for the whole article)


In [13]:
# Size sweep: how chunk size changes the number of chunks.
print(f'{"chunk_size":>10} | {"chunks":>7} | mean tokens')
print('-' * 34)
for size in (150, 300, 600):
    c = RecursiveChunker(tokenizer=chonkie_tok, chunk_size=size)
    ps = [x.text for x in c.chunk(text)]
    ts = [count_tokens(p) for p in ps]
    print(f'{size:>10} | {len(ps):>7} | {round(np.mean(ts),1)}')

chunk_size |  chunks | mean tokens
----------------------------------
       150 |      79 | 112.4
       300 |      34 | 259.6
       600 |      16 | 550.5


**Observation.** Smaller chunks → more of them (fine-grained, precise); larger chunks → fewer, each carrying more context. There is no universal best size — it is a knob you tune against your documents.


In [14]:
# Overlap demo: why chunks overlap, using a token chunker that supports it.
# (RecursiveChunker in chonkie 1.7.0 exposes no overlap parameter, so we use TokenChunker.)
overlap_chunker = TokenChunker(
    tokenizer=chonkie_tok,
    chunk_size=CONFIG['chunk_size_tokens'],
    chunk_overlap=CONFIG['chunk_overlap_tokens'],
)
overlap_passages = [c.text for c in overlap_chunker.chunk(text)]

print('chunks without overlap:', len(passages))
print('chunks with overlap   :', len(overlap_passages))
print()
print('end of chunk 0  :', repr(overlap_passages[0][-110:]))
print('start of chunk 1:', repr(overlap_passages[1][:110]))

chunks without overlap: 34
chunks with overlap   : 35

end of chunk 0  : 'tion, code execution capability, access to proprietary information sources and more.\nComponent One: Planning\nA'
start of chunk 1: '  - The agent learns to call external APIs for extra information that is missing from the model weights (often'


**Observation.** The tail of chunk 0 reappears at the head of chunk 1 — those are the shared overlap tokens. So a sentence that lands on a cut is not lost: it survives intact in one of the two neighbours. The cost is a little duplication, which is why overlap stays small (here, 50 of 300 tokens).


## 2.7 Vector store

A **vector store** saves each chunk together with its vector, forming a searchable library of our documents. We open a **persistent** Chroma collection (it lives on disk, so it survives restarts), then add the chunks.

Ids are stable (`<source_id>:<index>`), so re-running indexing **overwrites** rather than duplicating entries.


In [15]:
# Open (or create) the persistent collection.
client = chromadb.PersistentClient(path=CONFIG['store_path'])
collection = client.get_or_create_collection(
    name=CONFIG['collection'], metadata={'hnsw:space': 'cosine'}
)

print('items before adding:', collection.count())

items before adding: 34


In [16]:
# Embed every chunk and store it with its metadata.
vectors = model.encode(passages, normalize_embeddings=True)

collection.upsert(
    ids=[f'{source_id}:{i}' for i in range(len(passages))],
    documents=passages,
    embeddings=vectors.tolist(),
    metadatas=[{'source_id': source_id, 'chunk_index': i} for i in range(len(passages))],
)

print('items after adding:', collection.count())

items after adding: 34


In [17]:
# Peek: one stored chunk, to confirm text + metadata landed correctly.
first_id = collection.get(limit=1)
print('sample id      :', first_id['ids'])
print('sample metadata:', first_id['metadatas'])
print('sample text    :', first_id['documents'][0][:200].replace(chr(10), ' '), '...')

sample id      : ['lilianweng-agent:0']
sample metadata: [{'source_id': 'lilianweng-agent', 'chunk_index': 0}]
sample text    : Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The p ...


# Part 3 — Retrieval

Now the online half begins. To answer a question we embed it with the **same** embedding model, then ask the store for the chunks whose vectors are nearest — the "top-k".

A note on scores: Chroma returns a **cosine distance** (`1 - cosine_similarity`). So a *smaller* distance is a *better* match. Throughout, we print `score = 1 - distance` so that **higher is better**, matching the cosine-similarity convention from 2.4.


## 3.1 Embed the question and retrieve


In [18]:
query = 'What is Task Decomposition?'
query_vector = model.encode([query], normalize_embeddings=True).tolist()

hits = collection.query(query_embeddings=query_vector, n_results=CONFIG['k'])

for i, (cid, dist) in enumerate(zip(hits['ids'][0], hits['distances'][0]), start=1):
    print(f'[{i}] id={cid}  score={1 - dist:.3f}')

[1] id=lilianweng-agent:1  score=0.641
[2] id=lilianweng-agent:21  score=0.602
[3] id=lilianweng-agent:22  score=0.569
[4] id=lilianweng-agent:20  score=0.569
[5] id=lilianweng-agent:19  score=0.545


## 3.2 The passages that came back


In [19]:
# Show the actual passages that were retrieved.
for i, passage in enumerate(hits['documents'][0], start=1):
    print(f'--- chunk [{i}] ---')
    print(passage[:500])
    print()

--- chunk [1] ---
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the mode

--- chunk [2] ---
        "reasoning": "reasoning",
        "plan": "- short bulleted\n- list that conveys\n- long-term plan",
        "criticism": "constructive self-criticism",
        "speak": "thoughts summary to say to user"
    },
    "command": {
        "name": "command name",
        "args": {
            "arg name": "value"
        }
    }
}
Ensure the response can be parsed by Python json.loads
GPT-Engineer is another project to create a whole repository of code gi

Those are the chunks the model will be allowed to use. If the answer is not in them, no model can invent it — which is why **retrieval quality** matters more than model size.


## 3.3 Retrieval census — how many and from where?

Before trusting an answer, look at *what* came back: how many chunks, from how many sources, and how confident the ranking is. These tiny readouts are how you sanity-check a RAG system in practice.


In [20]:
# --- retrieval census ---
all_meta = collection.get(include=['metadatas'])['metadatas']
store_sources = {m.get('source_id') for m in all_meta if m}
hit_sources = [m.get('source_id') for m in hits['metadatas'][0] if m]

print('chunks in store     :', collection.count())
print('requested k         :', CONFIG['k'])
print('chunks returned     :', len(hits['documents'][0]))
print('distinct sources in store:', len(store_sources), sorted(store_sources))
print('distinct sources used    :', len(set(hit_sources)), sorted(set(hit_sources)))
print('chunks per source    :', dict(Counter(hit_sources)))

chunks in store     : 34
requested k         : 5
chunks returned     : 5
distinct sources in store: 1 ['lilianweng-agent']
distinct sources used    : 1 ['lilianweng-agent']
chunks per source    : {'lilianweng-agent': 5}


## 3.4 Score spread — confident or borderline?

Look at the gap between the best match and the worst. A steep drop means retrieval is confident; a flat spread means the matches are all mediocre and the answer may be weak.


In [21]:
scores = [1 - d for d in hits['distances'][0]]
print('top-k scores  :', [round(s, 3) for s in scores])
if len(scores) > 1:
    print(f'top-1 score   : {scores[0]:.3f}')
    print(f'gap (top-1 - top-k): {scores[0] - scores[-1]:.3f}')
    if scores[0] - scores[-1] < 0.1:
        print('-> flat spread: matches are all similar; the top match is not clearly better')
    else:
        print('-> steep drop: retrieval is confident about the top match')

top-k scores  : [0.641, 0.602, 0.569, 0.569, 0.545]
top-1 score   : 0.641
gap (top-1 - top-k): 0.096
-> flat spread: matches are all similar; the top match is not clearly better


## 3.5 k sweep — how many chunks do we need?

`k` is a trade-off. Retrieve too few and you may miss the answer; too many and you flood the model with loosely related text. Watch how the returned count and the scores change as `k` grows.


In [22]:
for k in (1, 3, 5, 10):
    h = collection.query(query_embeddings=query_vector, n_results=k)
    n = len(h['documents'][0])
    top = 1 - h['distances'][0][0]
    last = 1 - h['distances'][0][-1]
    print(f'k={k:>2}  returned={n:>2}  top_score={top:.3f}  kth_score={last:.3f}')

k= 1  returned= 1  top_score=0.641  kth_score=0.641
k= 3  returned= 3  top_score=0.641  kth_score=0.569
k= 5  returned= 5  top_score=0.641  kth_score=0.545
k=10  returned=10  top_score=0.641  kth_score=0.465


**Observation.** As `k` grows, the extra chunks have **lower** scores — they are the weaker matches. That's the trade-off in numbers: more context, but progressively less relevant.


# Part 4 — Generation

The last stage. The retrieved chunks go into a **prompt** next to the question, and the model writes an answer **from that context** rather than from memory. This grounding is the "G" in RAG.

The prompt asks for three things: answer only from the context, cite sources, and admit when the answer is not present.


## 4.1 Build the prompt


In [23]:
# Build the prompt: retrieved context + the question.
context = '\n\n'.join(hits['documents'][0])

prompt = f'''Answer the question based only on the following context:
{context}

Question: {query}
'''
print(prompt[:1200])

Answer the question based only on the following context:
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be don

## 4.2 Context budget — will this even fit?

The context window is a **shared budget**: the prompt, the retrieved context, and the answer all live inside it. Let's check we're not about to overflow it (and reserve room for the answer).


In [24]:
prompt_tokens = count_tokens(prompt)
answer_budget = CONFIG['max_answer_tokens']
used = prompt_tokens + answer_budget
window = CONFIG['context_window']

print(f'prompt tokens   : {prompt_tokens}')
print(f'answer reserved : {answer_budget}')
print(f'total used      : {used} / {window}  ({used/window*100:.2f}% of window)')
print('fits:', used <= window)

prompt tokens   : 1392
answer reserved : 1024
total used      : 2416 / 32768  (7.37% of window)
fits: True


**Observation.** Because we retrieve only `k` chunks, the prompt stays a tiny fraction of the window. This is why the `chunk_size × k` product must be kept in check — a very large `k` with large chunks could overflow the window and get truncated (silently losing the answer).


In [25]:
# Generate the answer with the local model.
reply = ollama.chat(
    model=CONFIG['llm_model'],
    options={'temperature': CONFIG['temperature']},
    messages=[{'role': 'user', 'content': prompt}],
)
answer = reply['message']['content']
print(answer)

{
  "thoughts": {
    "text": "Task Decomposition is the process of breaking a complex task into smaller, manageable sub‑tasks or steps so that an agent can understand, plan, and execute them more effectively. It often involves identifying subgoals, ordering them, and sometimes exploring multiple reasoning paths (e.g., via Chain of Thought or Tree of Thoughts).",
    "plan": "- Identify the main goal\n- Break it into sub‑goals\n- Sequence or parallelize sub‑tasks\n- Optionally explore alternative paths",
    "criticism": "I might have omitted specific examples of how Tree of Thoughts extends Chain of Thought, but the core definition is covered.",
    "speak": "Task Decomposition is breaking a big task into smaller steps."
  },
  "command": {
    "name": "do_nothing",
    "args": {}
  }
}


## 4.3 Citations

A grounded answer should be checkable. The model refers to the numbered context blocks as `[n]`; we map those back to the source each block came from, so the reader can verify every claim.


In [26]:
# Render the retrieved chunks as a numbered, cited source list.
print('Sources used in the answer:')
for i, (cid, meta, dist) in enumerate(
    zip(hits['ids'][0], hits['metadatas'][0], hits['distances'][0]), start=1
):
    sid = meta.get('source_id', 'unknown')
    print(f'  [{i}] score={1 - dist:.3f}  source={sid}  id={cid}')
print()
print('Any [n] in the answer above maps to the matching line here.')

Sources used in the answer:
  [1] score=0.641  source=lilianweng-agent  id=lilianweng-agent:1
  [2] score=0.602  source=lilianweng-agent  id=lilianweng-agent:21
  [3] score=0.569  source=lilianweng-agent  id=lilianweng-agent:22
  [4] score=0.569  source=lilianweng-agent  id=lilianweng-agent:20
  [5] score=0.545  source=lilianweng-agent  id=lilianweng-agent:19

Any [n] in the answer above maps to the matching line here.


## 4.4 When the context doesn't contain the answer

Grounding means the model should *decline* when the retrieved text doesn't support an answer. Let's check that it does — this is the failure mode that separates a trustworthy RAG system from a confident guesser.


In [27]:
# An unanswerable question: nothing in this article is about France.
off_topic = 'What is the capital of France?'
off_vector = model.encode([off_topic], normalize_embeddings=True).tolist()
off_hits = collection.query(query_embeddings=off_vector, n_results=CONFIG['k'])
off_context = '\n\n'.join(off_hits['documents'][0])
off_prompt = f'''Answer the question based only on the following context:
{off_context}

Question: {off_topic}
If the context does not contain the answer, say exactly: "I don't know from the provided context."
'''
off_reply = ollama.chat(
    model=CONFIG['llm_model'],
    options={'temperature': CONFIG['temperature']},
    messages=[{'role': 'user', 'content': off_prompt}],
)
print(off_reply['message']['content'])

I don't know from the provided context.


In [28]:
# Edge case: ask for more than the store holds.
huge = collection.query(query_embeddings=query_vector, n_results=999)
print('requested k=999 -> actually returned:', len(huge['documents'][0]),
      '(capped by what is in the store)')

requested k=999 -> actually returned: 34 (capped by what is in the store)


**Observations.**
- On the off-topic question, retrieval still returns chunks (they're the *closest*, not *correct*), but a grounded model should decline rather than invent. Watch that it does.
- Requesting `k=999` silently returns fewer — you get the whole store. Worth knowing so a `k` larger than your corpus doesn't confuse you.


# Part 5 — Putting it together

Everything above collapses into one plain function, exactly as the theory guide promised: `rag(question) -> answer`. Indexing stays as it is; the function does the serving half — embed, retrieve, prompt, generate.


## 5.1 The whole thing as one function


In [29]:
def rag(question: str, k: int = CONFIG['k']) -> str:
    query_vector = model.encode([question], normalize_embeddings=True).tolist()
    hits = collection.query(query_embeddings=query_vector, n_results=k)
    context = '\n\n'.join(hits['documents'][0])
    prompt = f'''Answer the question based only on the following context:
{context}

Question: {question}
'''
    reply = ollama.chat(
        model=CONFIG['llm_model'],
        options={'temperature': CONFIG['temperature']},
        messages=[{'role': 'user', 'content': prompt}],
    )
    return reply['message']['content']

## 5.2 Ask it a few questions


In [32]:
for q in [
    'What is Task Decomposition?',
    'How does reflection help an agent?',
    'What is a plan-and-execute agent?',
    'Who is Msawenkosi Magwaza?'
]:
    print('Q:', q)
    print(rag(q))
    print('-' * 80)

Q: What is Task Decomposition?
{
  "thoughts": {
    "text": "Task Decomposition is the process of breaking a complex task into smaller, manageable sub‑tasks or steps. It involves identifying sub‑goals, sequencing actions, and sometimes exploring multiple reasoning paths (e.g., Chain of Thought or Tree of Thoughts) to ensure all necessary components are addressed.",
    "plan": "- Identify the overall goal\n- List sub‑goals or components\n- Sequence the steps logically\n- Optionally generate alternative paths (Tree of Thoughts) and evaluate them",
    "criticism": "Make sure the decomposition covers all critical aspects and does not omit any necessary sub‑tasks.",
    "speak": "Task Decomposition means breaking a big task into smaller, easier steps."
  },
  "command": {
    "name": "task_complete",
    "args": {
      "reason": "Answer provided"
    }
  }
}
--------------------------------------------------------------------------------
Q: How does reflection help an agent?
{
  "though

## 5.3 Corpus census — what did we actually build?

One last look at the whole index: how many documents, chunks, vectors, and how they're distributed. This is the "state of the library" snapshot you'd log after an indexing run.


In [31]:
# Corpus census.
all_meta = collection.get(include=['metadatas'])['metadatas']
per_source = Counter(m.get('source_id', 'unknown') for m in all_meta if m)

print('documents (sources) :', len(per_source))
print('chunks in store     :', collection.count())
print('vectors in store    :', collection.count())
print('embedding dim       :', vectors.shape[1])
print('chunks per source   :', dict(per_source))
print('total indexed tokens:', sum(sizes))

documents (sources) : 1
chunks in store     : 34
vectors in store    : 34
embedding dim       : 1024
chunks per source   : {'lilianweng-agent': 34}
total indexed tokens: 8827


---

## What you've learned

You've walked the full RAG path and, at each step, **measured** it rather than just running it:

- **Indexing** — count tokens with the model's own tokenizer, embed so that meaning (not keywords) drives similarity, split on token boundaries, and observe chunk sizes, overlap, and a size sweep.
- **Retrieval** — see the top-k passages, then *count* them: chunks in store, chunks returned, distinct sources, per-source breakdown, score spread, and how `k` changes the picture.
- **Generation** — ground the answer in the context, check the context budget, render citations, and confirm the model *declines* when the context can't support an answer.

The natural next improvements, in order of impact: **better retrieval** (reranking, hybrid search), then **better chunking**, then a **larger model**. And whenever you change the embedding model, **re-embed everything** — the old vectors are unusable.
